## Importing the Required Libraries

In [1]:
import spacy
from spacy.matcher import Matcher

## Load the lightweight English NLP model

In [2]:
nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

## Define the Deontic Logic rules

In [3]:
obligation_pattern = [{"LOWER": {"IN": ["shall", "must", "will", "agrees"]}}]
permission_pattern = [{"LOWER": {"IN": ["may", "can", "reserves", "allowed"]}}]
prohibition_pattern = [{"LOWER": {"IN": ["shall", "must", "may"]}}, {"LOWER": "not"}]
condition_pattern = [{"LOWER": {"IN": ["subject", "except", "unless", "provided", "notwithstanding"]}}]

## Add rules to the matcher

In [4]:
matcher.add("OBLIGATION", [obligation_pattern])
matcher.add("PERMISSION", [permission_pattern])
matcher.add("PROHIBITION", [prohibition_pattern])
matcher.add("CONDITION", [condition_pattern])

def extract_deontic_metadata(clause_text):
    """Scans a clause and tags Deontic modalities and conditions."""
    doc = nlp(clause_text)
    matches = matcher(doc)
    metadata = {"Obligations": [], "Permissions": [], "Prohibitions": [], "Conditions": []}
    
    for match_id, start, end in matches:
        rule_type = nlp.vocab.strings[match_id]
        context_span = doc[start:min(end+5, len(doc))] 
        
        if rule_type == "OBLIGATION": metadata["Obligations"].append(context_span.text)
        elif rule_type == "PERMISSION": metadata["Permissions"].append(context_span.text)
        elif rule_type == "PROHIBITION": metadata["Prohibitions"].append(context_span.text)
        elif rule_type == "CONDITION": metadata["Conditions"].append(context_span.text)
            
    return metadata

## TESTING IT OUT

In [5]:
# Using one of the messy clauses that were just extracted from ABEMA.pdf
test_clause = "We may obtain the following information (hereinafter referred to as “User Information”) regarding the Service. (1) Personal information: The User's name, email address, date of birth and gender. (2) Information about the terminal used: Device, OS, the User agent, cookies, advertisement identifier \"IDFA/Android Ad ID\", terminal language setting, access country, etc. (3) Information on communication environment: telecommunications carriers, communication environments, IP addresses, access areas, etc. (4) Usage information: version of the Service, history of use of the Service, etc."

print("Analyzing Clause:")
print(f"'{test_clause}'\n")

tags = extract_deontic_metadata(test_clause)
for key, values in tags.items():
    if values: print(f"{key}: {values}")

Analyzing Clause:
'We may obtain the following information (hereinafter referred to as “User Information”) regarding the Service. (1) Personal information: The User's name, email address, date of birth and gender. (2) Information about the terminal used: Device, OS, the User agent, cookies, advertisement identifier "IDFA/Android Ad ID", terminal language setting, access country, etc. (3) Information on communication environment: telecommunications carriers, communication environments, IP addresses, access areas, etc. (4) Usage information: version of the Service, history of use of the Service, etc.'

Permissions: ['may obtain the following information (']
